# HyperBench Tutorial 07: End-to-End Benchmark Pipeline

This notebook brings the full HyperBench workflow together in one place.

The notation used throughout the tutorial series is:

- ground-truth hyperspectral image (GT HSI / HR HSI): `(H, W, C)`
- low-resolution hyperspectral image (LR HSI): `(H/r, W/r, C)`
- high-resolution multispectral image (HR MSI): `(H, W, c)`
- spectral response function (SRF): `(c, C)`
- point spread function (PSF): `(k, k)`

This notebook shows how to:

- define an entire benchmark from Python
- use explicit `degradation_specs` to select exact benchmark cases
- run a broader sweep by defining benchmark parameters
- save results to CSV
- flush each case to CSV as it finishes
- control overwrite behavior
- understand `fail_fast`
- inspect the structured benchmark rows directly

This notebook assumes the input scene already has spatial dimensions appropriate for the model being benchmarked.

## Related documentation

- `docs/index.md`
- `docs/quickstart.md`
- `docs/degradations.md`
- `docs/config-files.md`
- `docs/model-integration.md`
- `docs/adapters.md`

## Imports

In [1]:
from pathlib import Path
import csv

import cv2
import numpy as np

from hyperbench import (
    __version__,
    BenchmarkConfig,
    DegradationSpec,
    PipelineAdapter,
    load_hsi,
    normalize_image,
    print_data_stats,
    run_benchmark,
)

print("HyperBench version:", __version__)

HyperBench version: 0.1.0


## Scene configuration

This notebook assumes a MATLAB scene is available locally.

In [2]:
SCENE_PATH = Path("../data/DC_data.mat")
SCENE_KEY = "dc"

OUTPUT_DIR = Path("../notebook_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Scene path:", SCENE_PATH)
print("Scene key:", SCENE_KEY)

Scene path: ../data/DC_data.mat
Scene key: dc


## Load the scene once for inspection

`run_benchmark(...)` loads the scene internally from `scene_path`, but it is still useful to inspect the data directly at the start of the notebook.

In [3]:
scene = load_hsi(SCENE_PATH, key=SCENE_KEY)
gt_hsi = normalize_image(scene)

print_data_stats(gt_hsi, name="Ground-truth HSI")
print("GT HSI shape (H, W, C):", gt_hsi.shape)

Ground-truth HSI type: <class 'numpy.ndarray'>
Ground-truth HSI dtype: float32
Ground-truth HSI shape (H, W, B): (1280, 307, 191)
Ground-truth HSI min: 0.0
Ground-truth HSI max: 1.0
Ground-truth HSI mean: 0.2233569175004959
Ground-truth HSI std: 0.24737973511219025
GT HSI shape (H, W, C): (1280, 307, 191)


## A simple benchmarkable reconstruction method

The goal here is to demonstrate orchestration rather than reconstruction quality, so the notebook uses a small nearest-neighbor baseline.

It accepts the standard HyperBench pipeline inputs and returns a prediction with shape `(H, W, C)`.

In [4]:
class NearestResizePipeline:
    def __init__(self, name="NearestResizePipeline"):
        self.name = name

    def run_pipeline(self, HR_MSI, LR_HSI, srf, psf=None, metadata=None):
        target_h = int(HR_MSI.shape[0])
        target_w = int(HR_MSI.shape[1])

        _, _, C_hsi = LR_HSI.shape
        pred = np.empty((target_h, target_w, C_hsi), dtype=np.float32)

        for ch in range(C_hsi):
            pred[:, :, ch] = cv2.resize(
                LR_HSI[:, :, ch],
                (target_w, target_h),
                interpolation=cv2.INTER_NEAREST,
            )

        stats = {
            "framework": "numpy",
            "num_parameters": 0,
        }

        return pred, stats

## Wrap the method with `PipelineAdapter`

`PipelineAdapter` is the standard integration path for the `run_pipeline(...)` interface.

In [5]:
pipeline = NearestResizePipeline()

adapter = PipelineAdapter(
    pipeline=pipeline,
    name="NearestResizeBaseline",
    input_backend="numpy",
    output_backend="numpy",
    add_batch_dim=False,
    device="auto",
)

print("Adapter ready:", adapter.name)

Adapter ready: NearestResizeBaseline


## Part 1: Explicit benchmark design with `degradation_specs`

This approach is useful when the experiment set is known in advance and should be controlled exactly.

DegradationSpec allows a user to control the exact downsampling_ratio, msi_band_count, and the Signal to Noise ratios for noise injection into the lr_hsi and hr_msi. A user can specify the exact PSFs that they wish to run these degradations with when they build a benchmark configuration (shown below).

In [6]:
explicit_specs = [
    DegradationSpec(downsample_ratio=4,  msi_band_count=4,  spatial_snr_db=35.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=4,  spatial_snr_db=30.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=16, msi_band_count=4,  spatial_snr_db=25.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=32, msi_band_count=4,  spatial_snr_db=20.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=3,  spatial_snr_db=30.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=8,  spatial_snr_db=30.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=16, spatial_snr_db=30.0, spectral_snr_db=40.0),
]

for spec in explicit_specs:
    print(spec)

DegradationSpec(downsample_ratio=4, msi_band_count=4, spatial_snr_db=35.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=4, spatial_snr_db=30.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=16, msi_band_count=4, spatial_snr_db=25.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=32, msi_band_count=4, spatial_snr_db=20.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=3, spatial_snr_db=30.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=8, spatial_snr_db=30.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=16, spatial_snr_db=30.0, spectral_snr_db=40.0)


## Build an explicit `BenchmarkConfig`

This configuration uses:

- two PSFs
- one sigma
- one kernel radius
- exact `degradation_specs`
- clipping enabled
- CSV writing enabled
- per-case flushing enabled

In [7]:
explicit_csv_path = OUTPUT_DIR / "tutorial_explicit_results.csv"

explicit_config = BenchmarkConfig(
    scene_path=SCENE_PATH,
    scene_key=SCENE_KEY,
    psf_names=["gaussian", "moffat"],
    psf_sigmas=[3.4],
    psf_kernel_radii=[7],
    degradation_specs=explicit_specs,
    clip_prediction_to_unit_interval=True,
    prediction_clip_min=0.0,
    prediction_clip_max=1.0,
    output_csv_path=explicit_csv_path,
    save_csv=True,
    flush_csv_after_each_case=True,
    overwrite_csv_on_start=True,
    fail_fast=True,
)

print(explicit_config)

BenchmarkConfig(scene_path=PosixPath('../data/DC_data.mat'), scene_key='dc', psf_names=['gaussian', 'moffat'], psf_sigmas=[3.4], psf_kernel_radii=[7], degradation_specs=[DegradationSpec(downsample_ratio=4, msi_band_count=4, spatial_snr_db=35.0, spectral_snr_db=40.0), DegradationSpec(downsample_ratio=8, msi_band_count=4, spatial_snr_db=30.0, spectral_snr_db=40.0), DegradationSpec(downsample_ratio=16, msi_band_count=4, spatial_snr_db=25.0, spectral_snr_db=40.0), DegradationSpec(downsample_ratio=32, msi_band_count=4, spatial_snr_db=20.0, spectral_snr_db=40.0), DegradationSpec(downsample_ratio=8, msi_band_count=3, spatial_snr_db=30.0, spectral_snr_db=40.0), DegradationSpec(downsample_ratio=8, msi_band_count=8, spatial_snr_db=30.0, spectral_snr_db=40.0), DegradationSpec(downsample_ratio=8, msi_band_count=16, spatial_snr_db=30.0, spectral_snr_db=40.0)], msi_band_counts=[4], downsample_ratio_to_spatial_snr_db={4: 35.0}, spectral_snr_dbs=[40.0], fwhm_factor=4.2, seed=42, metrics=['rmse', 'psnr

## Why these CSV settings matter

This configuration uses:

- `save_csv=True`  
  results are written to disk

- `flush_csv_after_each_case=True`  
  each completed case is appended immediately

- `overwrite_csv_on_start=True`  
  an existing file at the same path is replaced at the start of the run

- `fail_fast=True`  
  the run stops immediately if a case raises an exception

For long experiments, `flush_csv_after_each_case=True` is particularly useful because partial results are preserved even if a later case fails.

## Run the explicit benchmark

In [8]:
explicit_results = run_benchmark(adapter, explicit_config)

print("Number of explicit result rows:", len(explicit_results.rows))
print("CSV written to:", explicit_csv_path.resolve())

2026-03-30 10:31:36.972576: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-30 10:31:37.006963: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-30 10:31:37.014102: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-30 10:31:37.032379: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-30 10:31:39.366371: W tensorflow/compiler/tf2

Number of explicit result rows: 14
CSV written to: /work/pi_mduarte_umass_edu/Ritik_Shah_Folder/hyperbench_dev/notebook_outputs/tutorial_explicit_results.csv


## Inspect the structured benchmark rows

Each row is a dictionary containing:

- scene and method identifiers
- degradation parameters
- input shapes
- metrics
- status information
- runtime
- optional stats returned by the pipeline

In [9]:
for i, row in enumerate(explicit_results.rows):
    print(f"Row {i}")
    for key, value in row.items():
        print(f"  {key}: {value}")
    print("-" * 100)

Row 0
  scene_name: DC_data
  shape_policy: strict
  method_name: NearestResizeBaseline
  psf_name: gaussian
  sigma: 3.4
  kernel_size: 15
  downsampling_ratio: 4
  msi_band_count: 4
  spatial_snr: 35.0
  spectral_snr: 40.0
  fwhm_factor: 4.2
  seed: 42
  gt_shape: (1280, 307, 191)
  lr_hsi_shape: (320, 76, 191)
  hr_msi_shape: (1280, 307, 4)
  prediction_clipped: True
  prediction_clip_min: 0.0
  prediction_clip_max: 1.0
  framework: numpy
  num_parameters: 0
  RMSE: 0.08892399757533938
  PSNR: 21.01359450329501
  SSIM: 0.6430274545097807
  UIQI: 0.9052680194428655
  ERGAS: 9.800908790916257
  SAM: 8.514435768127441
  status: success
  error: 
  runtime_seconds: 209.66523145791143
----------------------------------------------------------------------------------------------------
Row 1
  scene_name: DC_data
  shape_policy: strict
  method_name: NearestResizeBaseline
  psf_name: gaussian
  sigma: 3.4
  kernel_size: 15
  downsampling_ratio: 8
  msi_band_count: 4
  spatial_snr: 30.0
  s

## Read the CSV back from disk

This confirms what was written during the run.

In [10]:
with explicit_csv_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    explicit_csv_rows = list(reader)

print("CSV row count:", len(explicit_csv_rows))
print("CSV columns:")
for field in explicit_csv_rows[0].keys():
    print(" ", field)

CSV row count: 14
CSV columns:
  scene_name
  shape_policy
  method_name
  psf_name
  sigma
  kernel_size
  downsampling_ratio
  msi_band_count
  spatial_snr
  spectral_snr
  fwhm_factor
  seed
  gt_shape
  lr_hsi_shape
  hr_msi_shape
  prediction_clipped
  prediction_clip_min
  prediction_clip_max
  framework
  num_parameters
  RMSE
  PSNR
  SSIM
  UIQI
  ERGAS
  SAM
  status
  error
  runtime_seconds


## Print the CSV rows clearly

In [11]:
for i, row in enumerate(explicit_csv_rows):
    print(f"CSV row {i}")
    for key, value in row.items():
        print(f"  {key}: {value}")
    print("-" * 100)

CSV row 0
  scene_name: DC_data
  shape_policy: strict
  method_name: NearestResizeBaseline
  psf_name: gaussian
  sigma: 3.4
  kernel_size: 15
  downsampling_ratio: 4
  msi_band_count: 4
  spatial_snr: 35.0
  spectral_snr: 40.0
  fwhm_factor: 4.2
  seed: 42
  gt_shape: (1280, 307, 191)
  lr_hsi_shape: (320, 76, 191)
  hr_msi_shape: (1280, 307, 4)
  prediction_clipped: True
  prediction_clip_min: 0.0
  prediction_clip_max: 1.0
  framework: numpy
  num_parameters: 0
  RMSE: 0.08892399757533938
  PSNR: 21.01359450329501
  SSIM: 0.6430274545097807
  UIQI: 0.9052680194428655
  ERGAS: 9.800908790916257
  SAM: 8.514435768127441
  status: success
  error: 
  runtime_seconds: 209.66523145791143
----------------------------------------------------------------------------------------------------
CSV row 1
  scene_name: DC_data
  shape_policy: strict
  method_name: NearestResizeBaseline
  psf_name: gaussian
  sigma: 3.4
  kernel_size: 15
  downsampling_ratio: 8
  msi_band_count: 4
  spatial_snr: 

## Part 2: Sweep-style benchmark design

A sweep-style configuration is useful when you want HyperBench to build the experiment set automatically from parameter lists.

This style is appropriate when the benchmark should cover a broader grid of conditions.

In [12]:
sweep_csv_path = OUTPUT_DIR / "tutorial_sweep_results.csv"

sweep_config = BenchmarkConfig(
    scene_path=SCENE_PATH,
    scene_key=SCENE_KEY,
    psf_names=["gaussian", "moffat"],
    psf_sigmas=[3.4],
    psf_kernel_radii=[7],
    msi_band_counts=[3, 4, 8],
    downsample_ratio_to_spatial_snr_db={
        4: 35.0,
        8: 30.0,
    },
    spectral_snr_dbs=[40.0],
    clip_prediction_to_unit_interval=True,
    prediction_clip_min=0.0,
    prediction_clip_max=1.0,
    output_csv_path=sweep_csv_path,
    save_csv=True,
    flush_csv_after_each_case=True,
    overwrite_csv_on_start=True,
    fail_fast=True,
)

print(sweep_config)

BenchmarkConfig(scene_path=PosixPath('../data/DC_data.mat'), scene_key='dc', psf_names=['gaussian', 'moffat'], psf_sigmas=[3.4], psf_kernel_radii=[7], degradation_specs=None, msi_band_counts=[3, 4, 8], downsample_ratio_to_spatial_snr_db={4: 35.0, 8: 30.0}, spectral_snr_dbs=[40.0], fwhm_factor=4.2, seed=42, metrics=['rmse', 'psnr', 'ssim', 'uiqi', 'ergas', 'sam'], normalize_inputs=True, lower_percentile=1.0, upper_percentile=99.0, clip_prediction_to_unit_interval=True, prediction_clip_min=0.0, prediction_clip_max=1.0, save_csv=True, output_csv_path=PosixPath('../notebook_outputs/tutorial_sweep_results.csv'), flush_csv_after_each_case=True, overwrite_csv_on_start=True, fail_fast=True, user_srf=None, user_psf=None, metadata={})


## How the sweep is formed

In this sweep:

- PSFs: 2 values
- sigmas: 1 value
- kernel radii: 1 value
- downsampling ratios: 2 values
- MSI channel counts: 3 values
- spectral SNRs: 1 value

So the total number of benchmark cases is:

`2 × 1 × 1 × 2 × 3 × 1 = 12`

In [13]:
num_cases = (
    len(sweep_config.psf_names)
    * len(sweep_config.psf_sigmas)
    * len(sweep_config.psf_kernel_radii)
    * len(sweep_config.downsample_ratio_to_spatial_snr_db)
    * len(sweep_config.msi_band_counts)
    * len(sweep_config.spectral_snr_dbs)
)

print("Expected sweep case count:", num_cases)

Expected sweep case count: 12


## Run the sweep benchmark

In [14]:
sweep_results = run_benchmark(adapter, sweep_config)

print("Number of sweep result rows:", len(sweep_results.rows))
print("CSV written to:", sweep_csv_path.resolve())

[2026-03-30 11:19:08] INFO | hyperbench | Loading scene from '../data/DC_data.mat' with key 'dc'
[2026-03-30 11:19:10] INFO | hyperbench | Generating benchmark cases from config.
[2026-03-30 11:19:10] INFO | hyperbench | Initialized benchmark CSV at '../notebook_outputs/tutorial_sweep_results.csv'.
[2026-03-30 11:19:10] INFO | hyperbench | Starting benchmark run with 12 case(s) using adapter 'NearestResizeBaseline'.
[2026-03-30 11:19:10] INFO | hyperbench | Starting case_00000 | psf=gaussian, sigma=3.4, kernel=7, r=4, c=3, snr_spatial=35.0, snr_spectral=40.0, seed=42
[2026-03-30 11:22:29] INFO | hyperbench | Completed case_00000 successfully in 199.383s | RMSE=0.088924, PSNR=21.013595, SSIM=0.643027, UIQI=0.905268, ERGAS=9.800909, SAM=8.514436
[2026-03-30 11:22:29] INFO | hyperbench | Starting case_00001 | psf=gaussian, sigma=3.4, kernel=7, r=4, c=4, snr_spatial=35.0, snr_spectral=40.0, seed=42
[2026-03-30 11:25:46] INFO | hyperbench | Completed case_00001 successfully in 196.769s | RM

Number of sweep result rows: 12
CSV written to: /work/pi_mduarte_umass_edu/Ritik_Shah_Folder/hyperbench_dev/notebook_outputs/tutorial_sweep_results.csv


## Inspect the sweep rows

In [15]:
for i, row in enumerate(sweep_results.rows):
    print(f"Sweep row {i}")
    for key, value in row.items():
        print(f"  {key}: {value}")
    print("-" * 100)

Sweep row 0
  scene_name: DC_data
  shape_policy: strict
  method_name: NearestResizeBaseline
  psf_name: gaussian
  sigma: 3.4
  kernel_size: 15
  downsampling_ratio: 4
  msi_band_count: 3
  spatial_snr: 35.0
  spectral_snr: 40.0
  fwhm_factor: 4.2
  seed: 42
  gt_shape: (1280, 307, 191)
  lr_hsi_shape: (320, 76, 191)
  hr_msi_shape: (1280, 307, 3)
  prediction_clipped: True
  prediction_clip_min: 0.0
  prediction_clip_max: 1.0
  framework: numpy
  num_parameters: 0
  RMSE: 0.08892399757533938
  PSNR: 21.01359450329501
  SSIM: 0.6430274545097807
  UIQI: 0.9052680194428655
  ERGAS: 9.800908790916257
  SAM: 8.514435768127441
  status: success
  error: 
  runtime_seconds: 199.38252638559788
----------------------------------------------------------------------------------------------------
Sweep row 1
  scene_name: DC_data
  shape_policy: strict
  method_name: NearestResizeBaseline
  psf_name: gaussian
  sigma: 3.4
  kernel_size: 15
  downsampling_ratio: 4
  msi_band_count: 4
  spatial_s

## A compact view of the sweep cases

This view extracts a few columns that are usually the easiest to scan first.

In [16]:
keys_to_show = [
    "scene_name",
    "method_name",
    "psf_name",
    "downsampling_ratio",
    "msi_band_count",
    "spatial_snr",
    "spectral_snr",
    "RMSE",
    "PSNR",
    "SSIM",
    "SAM",
    "status",
]

for i, row in enumerate(sweep_results.rows):
    print(f"Case {i}")
    for key in keys_to_show:
        if key in row:
            print(f"  {key}: {row[key]}")
    print("-" * 80)

Case 0
  scene_name: DC_data
  method_name: NearestResizeBaseline
  psf_name: gaussian
  downsampling_ratio: 4
  msi_band_count: 3
  spatial_snr: 35.0
  spectral_snr: 40.0
  RMSE: 0.08892399757533938
  PSNR: 21.01359450329501
  SSIM: 0.6430274545097807
  SAM: 8.514435768127441
  status: success
--------------------------------------------------------------------------------
Case 1
  scene_name: DC_data
  method_name: NearestResizeBaseline
  psf_name: gaussian
  downsampling_ratio: 4
  msi_band_count: 4
  spatial_snr: 35.0
  spectral_snr: 40.0
  RMSE: 0.08892399757533938
  PSNR: 21.01359450329501
  SSIM: 0.6430274545097807
  SAM: 8.514435768127441
  status: success
--------------------------------------------------------------------------------
Case 2
  scene_name: DC_data
  method_name: NearestResizeBaseline
  psf_name: gaussian
  downsampling_ratio: 4
  msi_band_count: 8
  spatial_snr: 35.0
  spectral_snr: 40.0
  RMSE: 0.08892399757533938
  PSNR: 21.01359450329501
  SSIM: 0.6430274545

## `fail_fast` in practice

When `fail_fast=True`, the benchmark stops at the first exception.

When `fail_fast=False`, the failure is recorded and the benchmark continues to later cases.

The small example below illustrates the difference using a deliberately failing pipeline object.

In [17]:
class FailingPipeline:
    def run_pipeline(self, HR_MSI, LR_HSI, srf, psf=None, metadata=None):
        r = int(metadata["downsample_ratio"])
        if r == 8:
            raise RuntimeError("Intentional failure for demonstration.")

        target_h = int(metadata["target_shape"][0])
        target_w = int(metadata["target_shape"][1])

        _, _, C_hsi = LR_HSI.shape
        pred = np.empty((target_h, target_w, C_hsi), dtype=np.float32)

        for ch in range(C_hsi):
            pred[:, :, ch] = cv2.resize(
                LR_HSI[:, :, ch],
                (target_w, target_h),
                interpolation=cv2.INTER_NEAREST,
            )

        return pred

In [18]:
non_failfast_adapter = PipelineAdapter(
    pipeline=FailingPipeline(),
    name="FailingBaseline",
    input_backend="numpy",
    output_backend="numpy",
    add_batch_dim=False,
    device="auto",
)

non_failfast_csv_path = OUTPUT_DIR / "tutorial_failfast_false_results.csv"

non_failfast_config = BenchmarkConfig(
    scene_path=SCENE_PATH,
    scene_key=SCENE_KEY,
    psf_names=["gaussian"],
    psf_sigmas=[3.4],
    psf_kernel_radii=[7],
    degradation_specs=[
        DegradationSpec(downsample_ratio=4, msi_band_count=4, spatial_snr_db=35.0, spectral_snr_db=40.0),
        DegradationSpec(downsample_ratio=8, msi_band_count=4, spatial_snr_db=30.0, spectral_snr_db=40.0),
        DegradationSpec(downsample_ratio=16, msi_band_count=4, spatial_snr_db=25.0, spectral_snr_db=40.0),
    ],
    output_csv_path=non_failfast_csv_path,
    save_csv=True,
    flush_csv_after_each_case=True,
    overwrite_csv_on_start=True,
    fail_fast=False,
)

non_failfast_results = run_benchmark(non_failfast_adapter, non_failfast_config)

print("Rows recorded with fail_fast=False:", len(non_failfast_results.rows))
for row in non_failfast_results.rows:
    print(row["downsampling_ratio"], row["status"], row["error"])

[2026-03-30 11:59:22] INFO | hyperbench | Loading scene from '../data/DC_data.mat' with key 'dc'
[2026-03-30 11:59:23] INFO | hyperbench | Generating benchmark cases from config.
[2026-03-30 11:59:23] INFO | hyperbench | Initialized benchmark CSV at '../notebook_outputs/tutorial_failfast_false_results.csv'.
[2026-03-30 11:59:23] INFO | hyperbench | Starting benchmark run with 3 case(s) using adapter 'FailingBaseline'.
[2026-03-30 11:59:23] INFO | hyperbench | Starting case_00000 | psf=gaussian, sigma=3.4, kernel=7, r=4, c=4, snr_spatial=35.0, snr_spectral=40.0, seed=42
[2026-03-30 12:00:14] ERROR | hyperbench | Failed case_00000 after 51.520s | 'target_shape'
[2026-03-30 12:00:15] INFO | hyperbench | Starting case_00001 | psf=gaussian, sigma=3.4, kernel=7, r=8, c=4, snr_spatial=30.0, snr_spectral=40.0, seed=42
[2026-03-30 12:01:06] ERROR | hyperbench | Failed case_00001 after 51.348s | Intentional failure for demonstration.
[2026-03-30 12:01:06] INFO | hyperbench | Starting case_00002 

Rows recorded with fail_fast=False: 3
4 failed KeyError: 'target_shape'
8 failed RuntimeError: Intentional failure for demonstration.
16 failed KeyError: 'target_shape'


## Summary

This notebook showed the full orchestration layer of HyperBench from Python:

- building a benchmark with explicit `degradation_specs`
- building a broader parameter sweep
- running `run_benchmark(...)`
- saving results to CSV
- flushing rows incrementally
- controlling overwrite behavior
- understanding `fail_fast`
- inspecting structured result rows and CSV contents

This is the complete path from benchmark definition to recorded experiment output.